# Predict One Match With Simplified Version 2

This notebook uses the active Version 2 architecture:

- CatBoostClassifier for outcome probabilities
- rating-based expected goals
- Poisson score matrix
- candidate expected-points optimizer

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "version_2_ml" / "scripts").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "version_2_ml":
    PROJECT_ROOT = cwd.parent
elif cwd.name == "notebooks" and cwd.parent.name == "version_2_ml":
    PROJECT_ROOT = cwd.parents[1]
else:
    PROJECT_ROOT = cwd

sys.path.append(str(PROJECT_ROOT / "version_2_ml" / "scripts"))

from predict_single_match_v2 import (
    load_fixtures,
    load_elo,
    load_fifa,
    validate_fixture,
    build_feature_row,
    load_outcome_model,
    predict_outcome_probabilities,
    estimate_expected_goals,
    generate_score_matrix,
    generate_candidates,
    print_top_candidates,
    predict_single_match,
    save_prediction,
)


## Match Input

In [ ]:
team_1 = "Brazil"
team_2 = "Scotland"
stage = "Group Stage"
match_date = "2026-06-24"


## Load Data And Validate Fixture

In [ ]:
fixtures = load_fixtures()
elo = load_elo()
fifa = load_fifa()

fixture = validate_fixture(team_1, team_2, stage, match_date, fixtures, elo, fifa)
fixture


## Build Features

In [ ]:
feature_row = build_feature_row(fixture, elo, fifa)
feature_row


## Outcome Probabilities And Expected Goals

In [ ]:
outcome_model = load_outcome_model()
outcome_probabilities = predict_outcome_probabilities(outcome_model, feature_row)
expected_goals_team_1, expected_goals_team_2 = estimate_expected_goals(feature_row)

outcome_probabilities, expected_goals_team_1, expected_goals_team_2


## Score Matrix And Candidate Optimizer

In [ ]:
score_matrix = generate_score_matrix(expected_goals_team_1, expected_goals_team_2)
candidates = generate_candidates(score_matrix, outcome_probabilities)
print_top_candidates(candidates)
candidates.head(10)


## Final Prediction

In [ ]:
prediction = predict_single_match(team_1, team_2, stage, match_date)
save_prediction(prediction)
prediction.__dict__
